# Boeing Capstone Final Notebook and Reproducible Three-Statement Model

This combined notebook consolidates the historical diagnostics, statement integrity checks, scenario engine, run metadata, authoritative outputs, and final system checks for Boeing using the attached quarterly Excel workbook.

**Guardrails used throughout**
- All calculations reference the attached Boeing Excel data directly.
- Quarter-end data remain quarterly throughout; there are no yearly rollups or LTM substitutions.
- Signs are kept explicit and tied to the chapter conventions.
- Combined line items are handled carefully to avoid double counting or omission.
- Diagnostics and system checks are preserved at the end of the notebook rather than observed and discarded.


## Notebook map

1. Setup, file discovery, and reusable helpers  
2. Direct ingestion of Boeing quarterly statements from Excel  
3. Historical driver pack, common-size analysis, trends, ratio diagnostics, and exception flags  
4. Income statement, balance sheet, and cash flow integrity checks  
5. Free-cash-flow views and cross-checks  
6. Scenario registry and quarterly forecast engine  
7. Named scenario execution, authoritative outputs, and run manifest  
8. Two-way sensitivity and DSO break-even analysis  
9. Export pack plus final diagnostics and system checks


In [1]:

from pathlib import Path
from copy import deepcopy
from datetime import datetime
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

DISPLAY_N = 6
FORECAST_HORIZON_Q = 16
MODEL_VERSION = "boeing_capstone_combined_v1"
ALLOW_ONLINE_MARKET_DATA = False  # default False so the notebook runs cleanly offline
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

HARD_CODED_BA_MARKET_CAP_MM = {
    "2016-03-31": 81219.0,
    "2016-06-30": 81280.0,
    "2016-09-30": 81542.0,
    "2016-12-31": 96081.0,
    "2017-03-31": 107140.0,
    "2017-06-30": 117305.0,
    "2017-09-30": 150258.0,
    "2017-12-31": 175643.0,
    "2018-03-31": 192538.0,
    "2018-06-30": 195461.0,
    "2018-09-30": 213660.0,
    "2018-12-31": 183064.0,
    "2019-03-31": 215304.0,
    "2019-06-30": 204803.0,
    "2019-09-30": 214125.0,
    "2019-12-31": 183374.0,
    "2020-03-31": 84161.0,
    "2020-06-30": 103458.0,
    "2020-09-30": 93293.0,
    "2020-12-31": 124651.0,
    "2021-03-31": 148874.0,
    "2021-06-30": 140353.0,
    "2021-09-30": 129208.0,
    "2021-12-31": 118561.0,
    "2022-03-31": 113247.0,
    "2022-06-30": 81136.0,
    "2022-09-30": 72118.0,
    "2022-12-31": 113528.0,
    "2023-03-31": 127730.0,
    "2023-06-30": 127306.0,
    "2023-09-30": 115621.0,
    "2023-12-31": 157694.0,
    "2024-03-31": 118334.0,
    "2024-06-30": 112022.0,
    "2024-09-30": 93930.0,
    "2024-12-31": 132612.0,
    "2025-03-31": 128490.0,
    "2025-06-30": 157990.0,
    "2025-09-30": 163200.0,
    "2025-12-31": 165030.0,
}
HARD_CODED_BA_MARKET_CAP_SOURCE = "Quarter-end market cap series hard-coded for valuation outputs (USD millions)."

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def show_recent(obj, n=DISPLAY_N):
    if isinstance(obj, pd.Series):
        out = obj.tail(n).copy()
        try:
            out.index = pd.Index(pd.to_datetime(out.index).strftime("%Y-%m-%d"), name=out.index.name)
        except Exception:
            pass
        return out
    out = obj.tail(n).copy()
    try:
        out.index = pd.Index(pd.to_datetime(out.index).strftime("%Y-%m-%d"), name=out.index.name)
    except Exception:
        pass
    return out

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def safe_div(numer, denom):
    d = safe_num(denom)
    if isinstance(d, (pd.Series, pd.DataFrame)):
        d = d.replace(0, np.nan)
    else:
        d = np.nan if pd.isna(d) or d == 0 else d
    return safe_num(numer) / d

def pick_sheet(xls: pd.ExcelFile, keywords):
    names = list(xls.sheet_names)
    for name in names:
        if all(k.lower() in name.lower() for k in keywords):
            return name
    raise KeyError(f"Could not find sheet with keywords {keywords}. Available sheets: {names}")

def read_statement(xls: pd.ExcelFile, keywords):
    raw = pd.read_excel(xls, sheet_name=pick_sheet(xls, keywords))
    date_col = next((c for c in raw.columns if str(c).lower() in ["date", "period", "fiscaldateending", "fiscal_date_ending"]), None)
    if date_col is None:
        raise KeyError(f"Sheet for {keywords} is missing a recognized date column.")
    raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce").dt.normalize()
    raw = raw.dropna(subset=[date_col]).set_index(date_col).sort_index()

    numeric_cols = raw.select_dtypes(include="number").columns.tolist()
    scale_cols = [c for c in numeric_cols if not any(k in str(c).lower() for k in ["share", "shs", "eps", "per share"])]
    raw = raw.copy()
    raw[scale_cols] = raw[scale_cols].astype(float) / 1e6  # $ millions
    return raw

def get_series(df: pd.DataFrame, candidates, *, default=np.nan, required=False, name=None, combine=None, fallback_contains=None, allow_zero=True):
    cols = list(df.columns)
    low_map = {str(c).lower(): c for c in cols}

    def find(cands):
        for cand in cands:
            if cand in cols:
                return cand
            key = str(cand).lower()
            if key in low_map:
                return low_map[key]
        return None

    col = find(candidates)

    if col is None and fallback_contains:
        matches = [c for c in cols if any(sub.lower() in str(c).lower() for sub in fallback_contains)]
        if matches:
            col = max(matches, key=lambda c: float(safe_num(df[c]).abs().median(skipna=True) or 0.0))

    if col is not None:
        s = safe_num(df[col])
        if allow_zero or float(s.fillna(0.0).abs().sum()) > 0.0:
            s = s.fillna(default)
            s.name = name or col
            return s

    if combine:
        total = None
        for group in combine:
            c = find(group)
            if c is not None:
                s = safe_num(df[c]).fillna(0.0)
                total = s if total is None else total.add(s, fill_value=0.0)
        if total is not None:
            total.name = name or "combined"
            return total

    if required:
        raise KeyError(f"Missing required column {name or candidates} in columns: {cols}")
    return pd.Series(default, index=df.index, name=name or (candidates[0] if candidates else "value"))

def quarter_days(index: pd.DatetimeIndex) -> pd.Series:
    idx = pd.to_datetime(index)
    s = idx.to_series().diff().dt.days.astype(float)
    if len(s) > 0:
        s.iloc[0] = 91.0
    s.index = idx
    return s

DATA_CANDIDATES = [
    Path.cwd() / "BA_quarterly_statements.xlsx",
    Path.cwd() / "BA_quarterly_statements(1).xlsx",
    Path("/mnt/data/BA_quarterly_statements.xlsx"),
    Path("/mnt/data/BA_quarterly_statements(1).xlsx"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not locate the Boeing quarterly Excel file.")

COMPANY_NAME = "The Boeing Company"
NOTEBOOK_NAME = "Boeing_Capstone_Final_Combined_Notebook_Hardcoded_MarketCap.ipynb"
EXPORT_BUNDLE_NAME = "boeing_capstone_review_pack.xlsx"
AUTHORITATIVE_OUTPUT_NAME = "authoritative_outputs_long.csv"
RUN_MANIFEST_NAME = "run_manifest.csv"
SCENARIO_REGISTRY_NAME = "scenario_registry.csv"
CHANGE_LOG_NAME = "change_log.csv"


def file_sha256(path: Path, chunk_size: int = 1_048_576) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


DATA_FILE_SHA256 = file_sha256(DATA_PATH)
DATA_FILE_MODIFIED_TIMESTAMP = datetime.fromtimestamp(DATA_PATH.stat().st_mtime).isoformat(timespec="seconds")
DATA_VERSION = f"{DATA_PATH.name}|sha256={DATA_FILE_SHA256[:12]}|modified={DATA_FILE_MODIFIED_TIMESTAMP}"

RUN_METADATA = {
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "company": COMPANY_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "model_version": MODEL_VERSION,
    "data_file": DATA_PATH.name,
    "data_path": str(DATA_PATH),
    "data_version": DATA_VERSION,
    "data_file_sha256": DATA_FILE_SHA256,
    "data_file_modified_timestamp": DATA_FILE_MODIFIED_TIMESTAMP,
    "scenario_registry_reference": f"outputs/{SCENARIO_REGISTRY_NAME}",
    "run_manifest_reference": f"outputs/{RUN_MANIFEST_NAME}",
    "authoritative_output_reference": f"outputs/{AUTHORITATIVE_OUTPUT_NAME}",
    "export_bundle_reference": f"outputs/{EXPORT_BUNDLE_NAME}",
    "change_log_reference": f"outputs/{CHANGE_LOG_NAME}",
}

CHANGE_LOG = pd.DataFrame([
    {
        "change_id": "CL-001",
        "change_timestamp": RUN_METADATA["run_timestamp"],
        "change_area": "market_data",
        "change_summary": "Added hard-coded quarter-end Boeing market capitalization inputs in the setup cell so valuation outputs run offline with the attached Excel file.",
        "affected_artifacts": "Setup cell; historical ratios and valuation outputs",
        "change_type": "logic/input enhancement",
    },
    {
        "change_id": "CL-002",
        "change_timestamp": RUN_METADATA["run_timestamp"],
        "change_area": "run_control",
        "change_summary": "Expanded the run manifest to an explicit textbook schema with company, model version, data version, assumption-set identifier, scenario label, diagnostic status, and export references.",
        "affected_artifacts": "Run manifest; exported run manifest CSV; review pack RunManifest sheet",
        "change_type": "metadata enhancement",
    },
    {
        "change_id": "CL-003",
        "change_timestamp": RUN_METADATA["run_timestamp"],
        "change_area": "documentation",
        "change_summary": "Added a formal change log artifact and export so material notebook changes remain visible to a reviewer.",
        "affected_artifacts": "In-notebook change log display; exported change_log.csv; review pack ChangeLog sheet",
        "change_type": "documentation enhancement",
    },
    {
        "change_id": "CL-004",
        "change_timestamp": RUN_METADATA["run_timestamp"],
        "change_area": "assumptions",
        "change_summary": (
            "Replaced mechanically-extrapolated scenario registry with economically-grounded assumptions "
            "reflecting Boeing 2025 recovery trajectory. Key changes: gross_margin base lifted from -1% "
            "to +7% (grounded in 2025 full-year avg of +5.15% and clean-quarter avg of +11.6%); "
            "volume_growth base reduced from 3% to 1.5% per quarter to reflect realistic delivery ramp; "
            "opex_ratio base reduced from 8.0% to 7.5%; DIO base reduced from 374.6 to 350 days "
            "reflecting delivery ramp inventory drawdown. DSO and DPO retained from historical derivation. "
            "Scenario spreads (upside, downside, liquidity_stress) recalibrated to maintain economic "
            "coherence across the scenario set."
        ),
        "affected_artifacts": "scenario_registry; authoritative_outputs_long.csv; all downstream scenario outputs and dashboard",
        "change_type": "assumption revision",
    },
    {
        "change_id": "CL-005",
        "change_timestamp": RUN_METADATA["run_timestamp"],
        "change_area": "model_scope",
        "change_summary": (
            "Extended forecast horizon from 8 quarters (2 years) to 16 quarters (4 years, 2026-Q1 through "
            "2029-Q4). Scenario assumptions recalibrated for 4-year horizon: base gross_margin raised from "
            "7% to 8% (more recovery built into longer horizon); volume_growth base reduced from 1.5% to "
            "1.2% per quarter to reflect growth moderation as revenue base matures; DIO base reduced from "
            "350 to 340 days reflecting cumulative delivery ramp inventory drawdown over 4 years; upside "
            "gross_margin raised to 12% and volume_growth to 2.2% reflecting full 737 MAX ramp scenario."
        ),
        "affected_artifacts": "All scenario outputs, authoritative_outputs_long.csv, dashboard, write-up exhibits",
        "change_type": "model scope extension",
    },
])

pd.Series(RUN_METADATA, name="run_metadata")


run_timestamp                                                   2026-04-19T11:47:54
company                                                          The Boeing Company
notebook_name                     Boeing_Capstone_Final_Combined_Notebook_Hardco...
model_version                                           boeing_capstone_combined_v1
data_file                                              BA_quarterly_statements.xlsx
data_path                         C:\Users\pcswe\Fin Modeling\BA_quarterly_state...
data_version                      BA_quarterly_statements.xlsx|sha256=469a396bc6...
data_file_sha256                  469a396bc6e3fff3446b0e145f650d5dcfe45cbaa1450a...
data_file_modified_timestamp                                    2026-01-27T15:41:58
scenario_registry_reference                           outputs/scenario_registry.csv
run_manifest_reference                                     outputs/run_manifest.csv
authoritative_output_reference               outputs/authoritative_outputs_l

## 1) Load the Boeing quarterly statements and build the historical driver pack

This section normalizes the three statement tabs directly from the attached Excel workbook into one quarterly modeling table. The mappings are designed to stay reproducible for similarly structured company files, while still being explicit enough to avoid double counting combined lines.


In [2]:

xls = pd.ExcelFile(DATA_PATH)
income_q = read_statement(xls, ["income", "statement"])
balance_q = read_statement(xls, ["balance", "sheet"])
cashflow_q = read_statement(xls, ["cash", "flow"])

common_index = income_q.index.intersection(balance_q.index).intersection(cashflow_q.index)
income_q = income_q.loc[common_index].sort_index()
balance_q = balance_q.loc[common_index].sort_index()
cashflow_q = cashflow_q.loc[common_index].sort_index()

hist = pd.DataFrame(index=common_index)
hist.index.name = "Period"

# Income statement
hist["Revenue"] = get_series(income_q, ["revenue", "Revenue", "totalRevenue"], required=True, name="Revenue")
hist["COGS"] = get_series(income_q, ["costOfRevenue", "COGS", "cogs"], required=True, name="COGS").abs()
hist["GrossProfit_Reported"] = get_series(income_q, ["grossProfit", "GrossProfit"], default=np.nan, name="GrossProfit_Reported")
hist["R&D"] = get_series(income_q, ["researchAndDevelopmentExpenses"], default=0.0, name="R&D").abs()
hist["SGA_Selected"] = get_series(
    income_q,
    ["sellingGeneralAndAdministrativeExpenses"],
    default=np.nan,
    name="SGA_Selected",
    combine=[["generalAndAdministrativeExpenses"], ["sellingAndMarketingExpenses"]],
).abs()
hist["OperatingExpenses_Reported"] = get_series(income_q, ["operatingExpenses"], default=np.nan, name="OperatingExpenses_Reported").abs()
hist["D&A"] = get_series(income_q, ["depreciationAndAmortization"], default=np.nan, name="D&A").abs()
hist["InterestExpense"] = get_series(income_q, ["interestExpense"], default=np.nan, name="InterestExpense").abs()
hist["EBIT_Reported"] = get_series(income_q, ["ebit", "operatingIncome"], default=np.nan, name="EBIT_Reported")
hist["EBITDA_Reported"] = get_series(income_q, ["ebitda"], default=np.nan, name="EBITDA_Reported")
hist["EBT_Reported"] = get_series(income_q, ["incomeBeforeTax"], default=np.nan, name="EBT_Reported")
hist["TaxExpense"] = get_series(income_q, ["incomeTaxExpense"], default=np.nan, name="TaxExpense")
hist["NetIncome"] = get_series(income_q, ["netIncome"], default=np.nan, name="NetIncome")
hist["SharesDiluted"] = get_series(income_q, ["weightedAverageShsOutDil"], default=np.nan, name="SharesDiluted")

# Balance sheet
hist["Cash"] = get_series(balance_q, ["cashAndCashEquivalents", "cash"], default=np.nan, name="Cash")
hist["ShortTermInvestments"] = get_series(balance_q, ["shortTermInvestments"], default=0.0, name="ShortTermInvestments")
hist["AccountsReceivable"] = get_series(balance_q, ["accountsReceivables", "netReceivables"], default=np.nan, name="AccountsReceivable", fallback_contains=["receiv"])
hist["Inventory"] = get_series(balance_q, ["inventory"], default=np.nan, name="Inventory")
hist["AccountsPayable"] = get_series(balance_q, ["accountPayables", "totalPayables"], default=np.nan, name="AccountsPayable", fallback_contains=["payable", "accru"])
hist["CurrentAssets"] = get_series(balance_q, ["totalCurrentAssets"], default=np.nan, name="CurrentAssets")
hist["CurrentLiabilities"] = get_series(balance_q, ["totalCurrentLiabilities"], default=np.nan, name="CurrentLiabilities")
hist["PPENet"] = get_series(balance_q, ["propertyPlantEquipmentNet"], default=np.nan, name="PPENet")
hist["TotalAssets"] = get_series(balance_q, ["totalAssets"], default=np.nan, name="TotalAssets")
hist["TotalLiabilities"] = get_series(balance_q, ["totalLiabilities"], default=np.nan, name="TotalLiabilities")
hist["RetainedEarnings"] = get_series(balance_q, ["retainedEarnings"], default=np.nan, name="RetainedEarnings")
hist["CommonStock"] = get_series(balance_q, ["commonStock"], default=0.0, name="CommonStock")
hist["APIC"] = get_series(balance_q, ["additionalPaidInCapital"], default=0.0, name="APIC")
hist["TreasuryStock"] = get_series(balance_q, ["treasuryStock"], default=0.0, name="TreasuryStock")
hist["AOCI"] = get_series(balance_q, ["accumulatedOtherComprehensiveIncomeLoss"], default=0.0, name="AOCI")
hist["StockholdersEquity_Reported"] = get_series(balance_q, ["totalStockholdersEquity", "totalEquity"], default=np.nan, name="StockholdersEquity_Reported")
hist["MinorityInterest"] = get_series(balance_q, ["minorityInterest"], default=0.0, name="MinorityInterest")
hist["TotalLiabAndEquity_Reported"] = get_series(balance_q, ["totalLiabilitiesAndTotalEquity"], default=np.nan, name="TotalLiabAndEquity_Reported")
hist["TotalEquity"] = hist["TotalLiabAndEquity_Reported"] - hist["TotalLiabilities"]
hist["LongTermDebt"] = get_series(balance_q, ["longTermDebt"], default=0.0, name="LongTermDebt")
hist["ShortTermDebt"] = get_series(balance_q, ["shortTermDebt"], default=0.0, name="ShortTermDebt")
hist["TotalDebt"] = get_series(balance_q, ["totalDebt"], default=np.nan, name="TotalDebt")
hist.loc[hist["TotalDebt"].isna(), "TotalDebt"] = hist["LongTermDebt"] + hist["ShortTermDebt"]

# Cash flow
hist["CFO_Reported"] = get_series(cashflow_q, ["netCashProvidedByOperatingActivities", "operatingCashFlow"], default=np.nan, name="CFO_Reported")
hist["CFI_Reported"] = get_series(cashflow_q, ["netCashProvidedByInvestingActivities"], default=np.nan, name="CFI_Reported")
hist["CFF_Reported"] = get_series(cashflow_q, ["netCashProvidedByFinancingActivities"], default=np.nan, name="CFF_Reported")
hist["OtherCash_Reported"] = get_series(cashflow_q, ["effectOfForexChangesOnCash", "effectOfExchangeRateChangesOnCash"], default=0.0, name="OtherCash_Reported")
hist["NetChangeCash_Reported"] = get_series(cashflow_q, ["netChangeInCash"], default=np.nan, name="NetChangeCash_Reported")
hist["CashAtBeginning_Reported"] = get_series(cashflow_q, ["cashAtBeginningOfPeriod"], default=np.nan, name="CashAtBeginning_Reported")
hist["CashAtEnd_Reported"] = get_series(cashflow_q, ["cashAtEndOfPeriod"], default=np.nan, name="CashAtEnd_Reported")
hist["CapEx_Reported"] = get_series(cashflow_q, ["capitalExpenditure", "investmentsInPropertyPlantAndEquipment"], default=np.nan, name="CapEx_Reported").abs()
hist["DividendsPaid_Reported"] = get_series(cashflow_q, ["netDividendsPaid", "commonDividendsPaid", "dividendsPaid"], default=0.0, name="DividendsPaid_Reported").abs()
hist["NetDebtIssuance_Reported"] = get_series(cashflow_q, ["netDebtIssuance"], default=np.nan, name="NetDebtIssuance_Reported")
hist["SBC_Reported"] = get_series(cashflow_q, ["stockBasedCompensation"], default=0.0, name="SBC_Reported").abs()
hist["DeferredTax_Reported"] = get_series(cashflow_q, ["deferredIncomeTax"], default=0.0, name="DeferredTax_Reported")

print(f"Quarter count: {len(hist)}")
print(f"Date range: {hist.index.min().date()} to {hist.index.max().date()}")
display(show_recent(hist[[
    "Revenue","COGS","GrossProfit_Reported","EBITDA_Reported","NetIncome",
    "Cash","AccountsReceivable","Inventory","AccountsPayable","PPENet","TotalDebt","TotalEquity"
]]))


Quarter count: 40
Date range: 2016-03-31 to 2025-12-31


,Revenue,COGS,GrossProfit_Reported,EBITDA_Reported,NetIncome,Cash,AccountsReceivable,Inventory,AccountsPayable,PPENet,TotalDebt,TotalEquity
Period,,,,,,,,,,,,
2024-09-30,"17,840.00","21,346.00","-3,506.00","-5,052.00","-6,170.00","9,961.00","2,894.00","83,341.00","12,267.00","11,557.00","57,650.00","-23,562.00"
2024-12-31,"15,243.00","16,824.00","-1,581.00","-2,829.00","-3,865.00","13,801.00","2,625.00","87,550.00","11,364.00","11,726.00","54,188.00","-3,914.00"
2025-03-31,"19,496.00","17,064.00","2,432.00","1,250.00",-37.00,"10,142.00","3,204.00","89,077.00","11,034.00","11,767.00","53,618.00","-3,325.00"
2025-06-30,"22,749.00","20,299.00","2,450.00",609.00,-611.00,"7,087.00","3,190.00","87,853.00","11,238.00","11,976.00","53,323.00","-3,296.00"
2025-09-30,"23,270.00","25,645.00","-2,375.00","-4,014.00","-5,337.00","6,173.00","3,314.00","82,425.00","11,732.00","12,323.00","53,353.00","-8,253.00"
2025-12-31,"23,948.00","22,136.00","1,812.00",-439.00,"8,220.00","10,921.00","12,079.00","84,679.00","13,109.00","15,602.00","54,098.00","5,457.00"


## 2) Historical diagnostics: common-size, trends, ratios, valuation hooks, and exception flags

The ratio and trend layer stays quarterly from start to finish. Valuation multiples use the same market-cap acquisition logic used earlier in the semester, but default to an offline-safe path so the notebook still runs cleanly without external access.


In [3]:

ratio_base = hist.copy()
ratio_base["AvgAssets"] = (ratio_base["TotalAssets"] + ratio_base["TotalAssets"].shift(1)) / 2.0
ratio_base["AvgEquity"] = (ratio_base["TotalEquity"] + ratio_base["TotalEquity"].shift(1)) / 2.0
ratio_base["AvgAR"] = (ratio_base["AccountsReceivable"] + ratio_base["AccountsReceivable"].shift(1)) / 2.0
ratio_base["AvgInventory"] = (ratio_base["Inventory"] + ratio_base["Inventory"].shift(1)) / 2.0
ratio_base["AvgAP"] = (ratio_base["AccountsPayable"] + ratio_base["AccountsPayable"].shift(1)) / 2.0
hist_days = quarter_days(ratio_base.index)

common_size_is = pd.DataFrame(index=hist.index)
for col in ["Revenue", "COGS", "GrossProfit_Reported", "SGA_Selected", "R&D", "OperatingExpenses_Reported", "EBIT_Reported", "InterestExpense", "TaxExpense", "NetIncome"]:
    common_size_is[col] = safe_div(hist[col], hist["Revenue"]) * 100

common_size_bs = pd.DataFrame(index=hist.index)
for col in ["Cash", "AccountsReceivable", "Inventory", "PPENet", "CurrentAssets", "CurrentLiabilities", "TotalDebt", "TotalEquity"]:
    common_size_bs[col] = safe_div(hist[col], hist["TotalAssets"]) * 100

trend_df = pd.DataFrame(index=hist.index)
for col in ["Revenue", "COGS", "EBITDA_Reported", "NetIncome", "Cash", "TotalDebt"]:
    trend_df[f"{col}_YoY"] = hist[col].pct_change(4)
    trend_df[f"{col}_3yrCAGR"] = (safe_div(hist[col], hist[col].shift(12))).pow(1/3) - 1

ratios = pd.DataFrame(index=hist.index)
ratios["CurrentRatio"] = safe_div(hist["CurrentAssets"], hist["CurrentLiabilities"])
ratios["QuickRatio"] = safe_div(hist["Cash"] + hist["AccountsReceivable"], hist["CurrentLiabilities"])
ratios["CashRatio"] = safe_div(hist["Cash"], hist["CurrentLiabilities"])
ratios["GrossMargin"] = safe_div(hist["GrossProfit_Reported"], hist["Revenue"])
ratios["EBITMargin"] = safe_div(hist["EBIT_Reported"], hist["Revenue"])
ratios["NetMargin"] = safe_div(hist["NetIncome"], hist["Revenue"])
ratios["ROA"] = safe_div(hist["NetIncome"], ratio_base["AvgAssets"])
ratios["ROE"] = safe_div(hist["NetIncome"], ratio_base["AvgEquity"])
ratios["AssetTurnover"] = safe_div(hist["Revenue"], ratio_base["AvgAssets"])
ratios["Leverage"] = safe_div(ratio_base["AvgAssets"], ratio_base["AvgEquity"])
ratios["ROE_DuPont"] = ratios["NetMargin"] * ratios["AssetTurnover"] * ratios["Leverage"]
ratios["DSO"] = safe_div(ratio_base["AvgAR"], hist["Revenue"] / hist_days)
ratios["DIO"] = safe_div(ratio_base["AvgInventory"], hist["COGS"] / hist_days)
ratios["DPO"] = safe_div(ratio_base["AvgAP"], hist["COGS"] / hist_days)
ratios["CCC"] = ratios["DSO"] + ratios["DIO"] - ratios["DPO"]
ratios["DebtEBITDA"] = safe_div(hist["TotalDebt"], hist["EBITDA_Reported"])
ratios["NetDebtEBITDA"] = safe_div(hist["TotalDebt"] - hist["Cash"], hist["EBITDA_Reported"])
ratios["InterestCoverage"] = safe_div(hist["EBIT_Reported"], hist["InterestExpense"])

quarter_end_index = pd.to_datetime(hist.index).normalize()
market_cap_map = pd.Series(HARD_CODED_BA_MARKET_CAP_MM, dtype=float)
market_cap_map.index = pd.to_datetime(market_cap_map.index).normalize()

missing_market_cap = quarter_end_index.difference(market_cap_map.index)
if len(missing_market_cap) > 0:
    raise KeyError(f"Missing hard-coded market cap values for: {[d.strftime('%Y-%m-%d') for d in missing_market_cap]}")

shares_used_for_valuation = hist["SharesDiluted"].replace(0, np.nan).ffill().bfill()
if shares_used_for_valuation.isna().any():
    raise ValueError("SharesDiluted contains unresolved missing values; cannot compute PricePerShare from hard-coded market cap.")

hist["MarketCap"] = market_cap_map.reindex(quarter_end_index).to_numpy()
hist["PricePerShare"] = (hist["MarketCap"] * 1e6) / shares_used_for_valuation.to_numpy()
hist["EnterpriseValue"] = hist["MarketCap"] + hist["TotalDebt"] - hist["Cash"]
ratios["PE"] = safe_div(hist["MarketCap"], hist["NetIncome"])
ratios["EV_EBITDA"] = safe_div(hist["EnterpriseValue"], hist["EBITDA_Reported"])
ratios["EV_Sales"] = safe_div(hist["EnterpriseValue"], hist["Revenue"])

print("Market data view — last 6 quarters")
display(show_recent(hist[["PricePerShare", "SharesDiluted", "MarketCap", "EnterpriseValue"]].round(2)))

exception_report = pd.DataFrame(index=hist.index)
exception_report["QuickRatio_lt_1"] = ratios["QuickRatio"] < 1
exception_report["GrossMargin_lt_0"] = ratios["GrossMargin"] < 0
exception_report["InterestCoverage_lt_1"] = ratios["InterestCoverage"] < 1
exception_report["DebtEBITDA_gt_5"] = ratios["DebtEBITDA"] > 5
exception_report["ValuationDataMissing"] = hist["PricePerShare"].isna()

print("Common-size income statement (% of revenue) — last 6 quarters")
display(show_recent(common_size_is.round(2)))

print("\nRatios — last 6 quarters")
display(show_recent(ratios.round(2)))

print("\nException report — last 6 quarters")
display(show_recent(exception_report))


Market data view — last 6 quarters


,PricePerShare,SharesDiluted,MarketCap,EnterpriseValue
Period,,,,
2024-09-30,151.78,618856570,"93,930.00","141,619.00"
2024-12-31,184.67,718100000,"132,612.00","172,999.00"
2025-03-31,170.55,753400000,"128,490.00","171,966.00"
2025-06-30,208.82,756600000,"157,990.00","204,226.00"
2025-09-30,215.67,756700000,"163,200.00","210,380.00"
2025-12-31,207.56,795112414,"165,030.00","208,207.00"


Common-size income statement (% of revenue) — last 6 quarters


,Revenue,COGS,GrossProfit_Reported,SGA_Selected,R&D,OperatingExpenses_Reported,EBIT_Reported,InterestExpense,TaxExpense,NetIncome
Period,,,,,,,,,,
2024-09-30,100.00,119.65,-19.65,6.09,6.47,12.56,-30.81,4.08,-0.28,-34.59
2024-12-31,100.00,110.37,-10.37,9.21,5.48,14.70,-21.90,4.95,-1.52,-25.36
2025-03-31,100.00,87.53,12.47,5.66,4.33,9.99,4.02,3.63,0.55,-0.19
2025-06-30,100.00,89.23,10.77,7.89,4.00,11.89,0.65,3.12,0.22,-2.69
2025-09-30,100.00,110.21,-10.21,6.54,3.85,10.34,-19.36,2.98,0.60,-22.94
2025-12-31,100.00,92.43,7.57,6.94,4.03,10.97,-3.40,2.75,0.41,34.32



Ratios — last 6 quarters


,CurrentRatio,QuickRatio,CashRatio,GrossMargin,EBITMargin,NetMargin,ROA,ROE,AssetTurnover,Leverage,ROE_DuPont,DSO,DIO,DPO,CCC,DebtEBITDA,NetDebtEBITDA,InterestCoverage,PE,EV_EBITDA,EV_Sales
Period,,,,,,,,,,,,,,,,,,,,,
2024-09-30,1.12,0.13,0.10,-0.20,-0.31,-0.35,-0.04,0.30,0.13,-6.75,0.30,15.60,364.19,52.00,327.79,-11.41,-9.44,-7.55,-15.22,-28.03,7.94
2024-12-31,1.32,0.17,0.14,-0.10,-0.22,-0.25,-0.03,0.28,0.10,-10.70,0.28,16.66,467.25,64.61,419.29,-19.15,-14.28,-4.42,-34.31,-61.15,11.35
2025-03-31,1.23,0.13,0.10,0.12,0.04,-0.00,-0.00,0.01,0.12,-43.22,0.01,13.45,465.79,59.07,420.18,42.89,34.78,1.11,"-3,472.70",137.57,8.82
2025-06-30,1.23,0.10,0.07,0.11,0.01,-0.03,-0.00,0.18,0.15,-47.06,0.18,12.79,396.59,49.92,359.45,87.56,75.92,0.21,-258.58,335.35,8.98
2025-09-30,1.18,0.09,0.06,-0.10,-0.19,-0.23,-0.03,0.92,0.15,-26.42,0.92,12.86,305.43,41.20,277.09,-13.29,-11.75,-6.49,-30.58,-52.41,9.04
2025-12-31,1.27,0.21,0.10,0.08,-0.03,0.34,0.05,-5.88,0.15,-113.83,-5.88,29.57,347.25,51.62,325.20,-123.23,-98.35,-1.24,20.08,-474.28,8.69



Exception report — last 6 quarters


,QuickRatio_lt_1,GrossMargin_lt_0,InterestCoverage_lt_1,DebtEBITDA_gt_5,ValuationDataMissing
Period,,,,,
2024-09-30,True,True,True,False,False
2024-12-31,True,True,True,False,False
2025-03-31,True,False,False,True,False
2025-06-30,True,False,True,True,False
2025-09-30,True,True,True,False,False
2025-12-31,True,False,True,False,False


## 3) Financial statement integrity: income statement checks, roll-forwards, cash flow bridge, and free cash flow views

These are the mechanical checks that make the notebook reviewable:
- direct income-statement reconstruction checks,
- balance sheet tie-outs,
- cash roll-forward checks,
- PP&E, debt, and retained-earnings roll-forward residuals,
- a rebuilt CFO bridge, and
- FCFF / FCFE cross-checks.


In [4]:

statement_checks = pd.DataFrame(index=hist.index)
statement_checks["GrossProfit_Residual"] = (hist["Revenue"] - hist["COGS"]) - hist["GrossProfit_Reported"]
statement_checks["EBITDA_Residual"] = (hist["EBIT_Reported"] + hist["D&A"]) - hist["EBITDA_Reported"]
statement_checks["BS_Tie_Residual"] = hist["TotalAssets"] - hist["TotalLiabilities"] - hist["TotalEquity"]
statement_checks["CF_Roll_Residual_Detailed"] = hist["CashAtEnd_Reported"] - (
    hist["CashAtBeginning_Reported"] + hist["CFO_Reported"] + hist["CFI_Reported"] + hist["CFF_Reported"] + hist["OtherCash_Reported"]
)
statement_checks["CF_Roll_Residual_vs_NetChange"] = hist["CashAtEnd_Reported"] - (
    hist["CashAtBeginning_Reported"] + hist["NetChangeCash_Reported"]
)
statement_checks["PP&E_Roll_Residual"] = hist["PPENet"].shift(1) + hist["CapEx_Reported"] - hist["D&A"] - hist["PPENet"]
statement_checks["Debt_Roll_Residual"] = hist["TotalDebt"].shift(1) + hist["NetDebtIssuance_Reported"] - hist["TotalDebt"]
statement_checks["RE_Roll_Residual"] = hist["RetainedEarnings"].shift(1) + hist["NetIncome"] - hist["DividendsPaid_Reported"] - hist["RetainedEarnings"]

cfo_bridge = pd.DataFrame(index=hist.index)
cfo_bridge["NetIncome"] = hist["NetIncome"]
cfo_bridge["D&A"] = hist["D&A"]
cfo_bridge["SBC"] = hist["SBC_Reported"]
cfo_bridge["DeferredTax"] = hist["DeferredTax_Reported"]
cfo_bridge["dAR"] = hist["AccountsReceivable"].diff()
cfo_bridge["dInventory"] = hist["Inventory"].diff()
cfo_bridge["dAP"] = hist["AccountsPayable"].diff()
cfo_bridge["WC_CashImpact_3line"] = -(cfo_bridge["dAR"].fillna(0.0) + cfo_bridge["dInventory"].fillna(0.0)) + cfo_bridge["dAP"].fillna(0.0)
cfo_bridge["CFO_Rebuilt_3line"] = cfo_bridge["NetIncome"].fillna(0.0) + cfo_bridge["D&A"].fillna(0.0) + cfo_bridge["SBC"].fillna(0.0) + cfo_bridge["DeferredTax"].fillna(0.0) + cfo_bridge["WC_CashImpact_3line"]
cfo_bridge["CFO_Reported"] = hist["CFO_Reported"]
cfo_bridge["CFO_Rebuild_Residual"] = cfo_bridge["CFO_Rebuilt_3line"] - cfo_bridge["CFO_Reported"]

tax_rate_est = safe_div(hist["TaxExpense"].clip(lower=0.0), hist["EBT_Reported"].where(hist["EBT_Reported"] > 0)).clip(lower=0.0, upper=0.50).fillna(0.21)
d_owc = (hist["AccountsReceivable"] + hist["Inventory"] - hist["AccountsPayable"]).diff().fillna(0.0)
fcf_view = pd.DataFrame(index=hist.index)
fcf_view["EBIT"] = hist["EBIT_Reported"]
fcf_view["TaxRate_Est"] = tax_rate_est
fcf_view["NOPAT"] = hist["EBIT_Reported"] * (1.0 - tax_rate_est)
fcf_view["D&A"] = hist["D&A"]
fcf_view["CapEx"] = hist["CapEx_Reported"]
fcf_view["dOWC"] = d_owc
fcf_view["FCFF"] = fcf_view["NOPAT"] + fcf_view["D&A"] - fcf_view["CapEx"] - fcf_view["dOWC"]
fcf_view["AfterTaxInterest"] = hist["InterestExpense"] * (1.0 - tax_rate_est)
fcf_view["NetBorrowing"] = hist["NetDebtIssuance_Reported"].fillna(0.0)
fcf_view["FCFE_Bridge1"] = hist["NetIncome"].fillna(0.0) + hist["D&A"].fillna(0.0) - hist["CapEx_Reported"].fillna(0.0) - d_owc + fcf_view["NetBorrowing"]
fcf_view["FCFE_Bridge2"] = fcf_view["FCFF"] - fcf_view["AfterTaxInterest"] + fcf_view["NetBorrowing"]
fcf_view["FCFE_CrossCheck_Residual"] = fcf_view["FCFE_Bridge1"] - fcf_view["FCFE_Bridge2"]

print("Statement integrity checks — last 6 quarters")
display(show_recent(statement_checks.round(2)))

print("\nCFO rebuild bridge — last 6 quarters")
display(show_recent(cfo_bridge.round(2)))

print("\nFCFF / FCFE view — last 6 quarters")
display(show_recent(fcf_view.round(2)))


Statement integrity checks — last 6 quarters


,GrossProfit_Residual,EBITDA_Residual,BS_Tie_Residual,CF_Roll_Residual_Detailed,CF_Roll_Residual_vs_NetChange,PP&E_Roll_Residual,Debt_Roll_Residual,RE_Roll_Residual
Period,,,,,,,,
2024-09-30,0.00,0.00,0.00,0.00,0.00,-414.00,-35.00,0.00
2024-12-31,0.00,0.00,0.00,0.00,0.00,-30.00,-346.00,72.00
2025-03-31,0.00,0.00,0.00,0.00,0.00,167.00,304.00,14.00
2025-06-30,0.00,0.00,0.00,0.00,0.00,-242.00,-18.00,0.00
2025-09-30,0.00,0.00,0.00,0.00,0.00,-164.00,-34.00,1.00
2025-12-31,0.00,0.00,0.00,0.00,0.00,"-3,228.00",-162.00,-1.00



CFO rebuild bridge — last 6 quarters


,NetIncome,D&A,SBC,DeferredTax,dAR,dInventory,dAP,WC_CashImpact_3line,CFO_Rebuilt_3line,CFO_Reported,CFO_Rebuild_Residual
Period,,,,,,,,,,,
2024-09-30,"-6,170.00",444.00,102.00,0.00,-261.00,"-2,320.00",403.00,"2,984.00","-2,640.00","-1,345.00","-1,295.00"
2024-12-31,"-3,865.00",509.00,97.00,-0.15,-269.00,"4,209.00",-903.00,"-4,843.00","-8,102.15","-3,450.00","-4,652.15"
2025-03-31,-37.00,466.00,135.00,0.00,579.00,"1,527.00",-330.00,"-2,436.00","-1,872.00","-1,616.00",-256.00
2025-06-30,-611.00,460.00,119.00,0.00,-14.00,"-1,224.00",204.00,"1,442.00","1,410.00",227.00,"1,183.00"
2025-09-30,"-5,337.00",491.00,89.00,0.00,124.00,"-5,428.00",494.00,"5,798.00","1,041.00","1,123.00",-82.00
2025-12-31,"8,220.00",376.00,83.00,0.00,"8,765.00","2,254.00","1,377.00","-9,642.00",-963.00,"1,331.00","-2,294.00"



FCFF / FCFE view — last 6 quarters


,EBIT,TaxRate_Est,NOPAT,D&A,CapEx,dOWC,FCFF,AfterTaxInterest,NetBorrowing,FCFE_Bridge1,FCFE_Bridge2,FCFE_CrossCheck_Residual
Period,,,,,,,,,,,,
2024-09-30,"-5,496.00",0.21,"-4,341.84",444.00,611.00,"-2,984.00","-1,524.84",575.12,-312.00,"-3,665.00","-2,411.96","-1,253.04"
2024-12-31,"-3,338.00",0.21,"-2,637.02",509.00,648.00,"4,843.00","-7,619.02",596.45,"-3,808.00","-12,655.00","-12,023.47",-631.53
2025-03-31,784.00,0.50,392.00,466.00,674.00,"2,436.00","-2,252.00",354.00,-266.00,"-2,947.00","-2,872.00",-75.00
2025-06-30,149.00,0.21,117.71,460.00,427.00,"-1,442.00","1,592.71",560.90,-313.00,551.00,718.81,-167.81
2025-09-30,"-4,505.00",0.21,"-3,558.95",491.00,674.00,"-5,798.00","2,056.05",548.26,-4.00,274.00,"1,503.79","-1,229.79"
2025-12-31,-815.00,0.01,-805.30,376.00,427.00,"9,642.00","-10,498.30",651.16,583.00,-890.00,"-10,566.46","9,676.46"


## 4) Scenario registry and quarterly forecast engine

The base case is inferred directly from the most recent six quarters of Boeing history. Named scenarios then change assumptions, not formulas.


In [5]:

last6 = hist.tail(DISPLAY_N).copy()
last6_days = quarter_days(last6.index)
gross_margin_hist = 1.0 - safe_div(last6["COGS"], last6["Revenue"])
ebitda_margin_hist = safe_div(last6["EBITDA_Reported"], last6["Revenue"])
opex_ratio_hist = (gross_margin_hist - ebitda_margin_hist).clip(lower=0.0)
dso_hist = safe_div(last6["AccountsReceivable"], last6["Revenue"] / last6_days)
dio_hist = safe_div(last6["Inventory"], last6["COGS"] / last6_days)
dpo_hist = safe_div(last6["AccountsPayable"], last6["COGS"] / last6_days)
capex_ratio_hist = safe_div(last6["CapEx_Reported"], last6["Revenue"]).clip(lower=0.0)
dep_rate_hist = safe_div(last6["D&A"], last6["PPENet"].shift(1).replace(0, np.nan)).fillna(0.03).clip(lower=0.0, upper=0.20)
avg_debt_hist = (last6["TotalDebt"].shift(1).fillna(last6["TotalDebt"]) + last6["TotalDebt"]) / 2.0
REF_RATE_ANNUAL_DEFAULT = 0.04
borrow_spread_hist = (safe_div(last6["InterestExpense"] * 4.0, avg_debt_hist) - REF_RATE_ANNUAL_DEFAULT).clip(lower=0.0)
tax_hist = safe_div(last6["TaxExpense"].clip(lower=0.0), last6["EBT_Reported"].where(last6["EBT_Reported"] > 0)).clip(lower=0.0, upper=0.50)
div_payout_hist = safe_div(last6["DividendsPaid_Reported"].clip(lower=0.0), last6["NetIncome"].clip(lower=0.0))

base_assumptions = {
    "volume_growth": float(last6["Revenue"].pct_change().median(skipna=True) or 0.0),
    "price_growth": 0.0,
    "gross_margin": float(gross_margin_hist.median(skipna=True)),
    "opex_ratio": float(opex_ratio_hist.median(skipna=True)),
    "dso": float(dso_hist.median(skipna=True)),
    "dio": float(dio_hist.median(skipna=True)),
    "dpo": float(dpo_hist.median(skipna=True)),
    "capex_to_sales": float(capex_ratio_hist.median(skipna=True)),
    "dep_rate_q": float(dep_rate_hist.median(skipna=True)),
    "ref_rate_annual": float(REF_RATE_ANNUAL_DEFAULT),
    "borrowing_spread_annual": float(borrow_spread_hist.median(skipna=True) if not borrow_spread_hist.isna().all() else 0.025),
    "tax_rate": float(tax_hist.median(skipna=True) if not tax_hist.isna().all() else 0.21),
    "dividend_payout": float(div_payout_hist.median(skipna=True) if not div_payout_hist.isna().all() else 0.0),
    "revolver_limit": 15000.0,
    "cash_floor": 0.0,
    "mandatory_amortization_q": 0.0,
}

# ── Scenario registry — CL-004, CL-005 ──────────────────────────────────────
# Assumptions recalibrated for 16-quarter (4-year) forecast horizon.
#
# Key grounding references:
#   Gross margin — 2025 full-year avg: +5.15%; clean H1 2025 avg: +11.62%
#     Base raised to 8% (vs 7% for 2-year) reflecting more recovery over 4 years.
#     Upside at 12% approaches Boeing's pre-crisis normalized level.
#   Volume growth — base 1.2%/Q (~5% annual) reflects mature ramp phase;
#     upside 2.2%/Q reflects accelerated 737 MAX + 777X delivery ramp.
#     Growth moderates from 2-year assumptions as revenue base grows.
#   DIO — base 340 days (vs 350) reflects cumulative inventory drawdown
#     as deliveries ramp over a full 4-year horizon.
#   All other drivers retained from CL-004 derivation.
#
# Scenario economic narratives:
#   Base: Steady recovery, 737 MAX at ~38/month by 2028, defense charges behind.
#   Upside: 737 MAX + 777X ramp, strong international orders, margin normalization.
#   Downside: Persistent delivery delays, quality issues, modest contraction.
#   Liquidity Stress: New grounding event or major charge forces revolver drawdown.
# ──────────────────────────────────────────────────────────────────────────────

scenario_registry = pd.DataFrame({
    "volume_growth": {
        "base":              0.012,   # ~5% annual; mature ramp over 4 years
        "downside":         -0.005,   # Slight contraction; delivery delays
        "upside":            0.022,   # ~9% annual; 737 MAX + 777X ramp
        "liquidity_stress": -0.015,   # Production cuts under severe cash pressure
    },
    "price_growth": {
        "base":              0.000,
        "downside":         -0.010,
        "upside":            0.010,
        "liquidity_stress": -0.015,
    },
    "gross_margin": {
        # Base: 8% reflects continued recovery from 2025 levels (5.15% FY avg).
        # 4-year horizon allows more normalization than 2-year model.
        # Upside: 12% approaches pre-crisis normalized margins.
        # Downside: 3% — partial recovery stalled by charges/quality issues.
        # Stress: -3% — new writedowns hit; gross margin turns negative again.
        "base":              0.080,
        "downside":          0.030,
        "upside":            0.120,
        "liquidity_stress": -0.030,
    },
    "opex_ratio": {
        # Q1 2025: 6.1%, Q2 2025: 8.1%. Base 7.5% is conservative.
        # Upside benefits from operating leverage on higher revenue.
        "base":              0.075,
        "downside":          0.085,
        "upside":            0.070,
        "liquidity_stress":  0.100,
    },
    "dso": {
        # Grounded in trade AR (accountsReceivables) / (Revenue/91).
        # Boeing collects quickly; 13-17 day historical range.
        "base":              base_assumptions["dso"],
        "downside":          base_assumptions["dso"] + 8.0,
        "upside":            max(base_assumptions["dso"] - 3.0, 0.0),
        "liquidity_stress":  base_assumptions["dso"] + 15.0,
    },
    "dio": {
        # Actual Q4 2025 DIO: 348 days. Base 340 reflects 4-year delivery ramp.
        # Upside: 300 — significant backlog clearance over full horizon.
        # Stress: 410 — deliveries slip, inventory compounds.
        "base":              340.0,
        "downside":          380.0,
        "upside":            300.0,
        "liquidity_stress":  410.0,
    },
    "dpo": {
        "base":              base_assumptions["dpo"],
        "downside":          max(base_assumptions["dpo"] - 5.0, 0.0),
        "upside":            base_assumptions["dpo"] + 3.0,
        "liquidity_stress":  max(base_assumptions["dpo"] - 8.0, 0.0),
    },
    "capex_to_sales": {
        "base":              base_assumptions["capex_to_sales"],
        "downside":          max(base_assumptions["capex_to_sales"] - 0.01, 0.0),
        "upside":            base_assumptions["capex_to_sales"] + 0.01,
        "liquidity_stress":  base_assumptions["capex_to_sales"],
    },
    "borrowing_spread_annual": {
        "base":              base_assumptions["borrowing_spread_annual"],
        "downside":          base_assumptions["borrowing_spread_annual"] + 0.01,
        "upside":            max(base_assumptions["borrowing_spread_annual"] - 0.0025, 0.0),
        "liquidity_stress":  base_assumptions["borrowing_spread_annual"] + 0.02,
    },
    "cash_floor": {
        "base":              base_assumptions["cash_floor"],
        "downside":          base_assumptions["cash_floor"],
        "upside":            base_assumptions["cash_floor"],
        "liquidity_stress":  max(5000.0, base_assumptions["cash_floor"]),
    },
    "revolver_limit": {
        "base": 15000.0, "downside": 15000.0,
        "upside": 15000.0, "liquidity_stress": 15000.0,
    },
}, index=["base", "downside", "upside", "liquidity_stress"])

def make_assumptions(base: dict, scenario_name: str, registry: pd.DataFrame) -> dict:
    a = deepcopy(base)
    a.update(registry.loc[scenario_name].dropna().to_dict())
    return a

display(pd.Series(base_assumptions, name="base_assumptions"))
print("\nScenario registry")
display(scenario_registry)


volume_growth                   0.03
price_growth                    0.00
gross_margin                   -0.01
opex_ratio                      0.08
dso                            14.78
dio                           374.57
dpo                            53.39
capex_to_sales                  0.03
dep_rate_q                      0.04
ref_rate_annual                 0.04
borrowing_spread_annual         0.01
tax_rate                        0.26
dividend_payout                 0.01
revolver_limit             15,000.00
cash_floor                      0.00
mandatory_amortization_q        0.00
Name: base_assumptions, dtype: float64


Scenario registry


,volume_growth,price_growth,gross_margin,opex_ratio,dso,dio,dpo,capex_to_sales,borrowing_spread_annual,cash_floor,revolver_limit
base,0.01,0.00,0.08,0.07,14.78,340.00,53.39,0.03,0.01,0.00,"15,000.00"
downside,-0.01,-0.01,0.03,0.09,22.78,380.00,48.39,0.02,0.02,0.00,"15,000.00"
upside,0.02,0.01,0.12,0.07,11.78,300.00,56.39,0.04,0.01,0.00,"15,000.00"
liquidity_stress,-0.01,-0.01,-0.03,0.10,29.78,410.00,45.39,0.03,0.03,"5,000.00","15,000.00"


## 5) Run the forecast engine, preserve run records, and build the authoritative output layer


In [6]:

def future_quarters(last_q: pd.Timestamp, n_q: int) -> pd.DatetimeIndex:
    return pd.date_range(last_q, periods=n_q + 1, freq=pd.offsets.QuarterEnd())[1:]

def forecast_days(periods: pd.DatetimeIndex) -> pd.Series:
    s = pd.Series(pd.to_datetime(periods)).diff().dt.days.astype(float)
    if len(s) > 0:
        s.iloc[0] = 91.0
    s.index = pd.Index(periods)
    return s

def run_three_statement_forecast(hist: pd.DataFrame, assumptions: dict, *, horizon_q: int) -> dict:
    last_q = hist.index.max()
    periods = future_quarters(last_q, horizon_q)
    days_q = forecast_days(periods)

    anchor = hist.loc[last_q]
    rev_prev = float(anchor["Revenue"])
    cash_begin = float(anchor["Cash"])
    ar_begin = float(anchor["AccountsReceivable"])
    inv_begin = float(anchor["Inventory"])
    ap_begin = float(anchor["AccountsPayable"])
    ppe_begin = float(anchor["PPENet"])
    total_debt = float(anchor["TotalDebt"])
    term_begin = float(anchor["LongTermDebt"]) if pd.notna(anchor["LongTermDebt"]) else total_debt
    revl_begin = float(anchor["ShortTermDebt"]) if pd.notna(anchor["ShortTermDebt"]) else max(total_debt - term_begin, 0.0)
    total_eq = float(anchor["TotalEquity"])
    re_begin = float(anchor["RetainedEarnings"])
    common_capital = float(total_eq - re_begin)
    total_assets = float(anchor["TotalAssets"])
    total_liab = float(anchor["TotalLiabilities"])

    other_assets = max(total_assets - (cash_begin + ar_begin + inv_begin + ppe_begin), 0.0)
    other_liab = max(total_liab - (ap_begin + term_begin + revl_begin), 0.0)

    rows = []
    for period, day_count in days_q.items():
        rev_t = rev_prev * (1.0 + assumptions["volume_growth"]) * (1.0 + assumptions["price_growth"])
        cogs_t = rev_t * (1.0 - assumptions["gross_margin"])
        gross_profit_t = rev_t - cogs_t
        ebitda_t = rev_t * (assumptions["gross_margin"] - assumptions["opex_ratio"])
        dep_t = assumptions["dep_rate_q"] * ppe_begin
        ebit_t = ebitda_t - dep_t
        interest_t = ((assumptions["ref_rate_annual"] + assumptions["borrowing_spread_annual"]) / 4.0) * (term_begin + revl_begin)
        ebt_t = ebit_t - interest_t
        tax_t = max(ebt_t, 0.0) * assumptions["tax_rate"]
        ni_t = ebt_t - tax_t
        div_t = max(ni_t, 0.0) * assumptions["dividend_payout"]

        rev_per_day = rev_t / max(day_count, 1.0)
        cogs_per_day = cogs_t / max(day_count, 1.0)
        ar_end = rev_per_day * assumptions["dso"]
        inv_end = cogs_per_day * assumptions["dio"]
        ap_end = cogs_per_day * assumptions["dpo"]

        d_ar = ar_end - ar_begin
        d_inv = inv_end - inv_begin
        d_ap = ap_end - ap_begin

        cfo_t = ni_t + dep_t - d_ar - d_inv + d_ap
        capex_t = max(assumptions["capex_to_sales"] * rev_t, 0.0)
        cfi_t = -capex_t

        mandatory_amort = min(assumptions.get("mandatory_amortization_q", 0.0), term_begin)
        cff_before_revolver = -mandatory_amort - div_t
        cash_prefin = cash_begin + cfo_t + cfi_t + cff_before_revolver

        draw = 0.0
        repay = 0.0
        if cash_prefin < assumptions["cash_floor"]:
            draw = min(assumptions["cash_floor"] - cash_prefin, max(assumptions["revolver_limit"] - revl_begin, 0.0))
            cash_end = cash_prefin + draw
            revl_end = revl_begin + draw
        else:
            cash_end = cash_prefin
            revl_end = revl_begin

        term_end = term_begin - mandatory_amort
        total_debt_end = term_end + revl_end

        ppe_end = ppe_begin + capex_t - dep_t
        re_end = re_begin + ni_t - div_t
        total_assets_end = cash_end + ar_end + inv_end + ppe_end + other_assets
        total_liab_end = ap_end + term_end + revl_end + other_liab
        total_eq_end = common_capital + re_end
        bs_resid = total_assets_end - total_liab_end - total_eq_end

        cff_t = cff_before_revolver + draw - repay
        net_change_cash_t = cfo_t + cfi_t + cff_t
        d_owc_t = (ar_end + inv_end - ap_end) - (ar_begin + inv_begin - ap_begin)
        fcff_t = ebit_t * (1.0 - assumptions["tax_rate"]) + dep_t - capex_t - d_owc_t
        after_tax_interest_t = interest_t * (1.0 - assumptions["tax_rate"])
        fcfe_t = fcff_t - after_tax_interest_t + (draw - repay - mandatory_amort)
        net_debt_t = total_debt_end - cash_end
        nde_annual = np.nan if ebitda_t <= 1.0 else net_debt_t / (ebitda_t * 4.0)

        rows.append({
            "Period": period,
            "Revenue": rev_t,
            "COGS": cogs_t,
            "GrossProfit": gross_profit_t,
            "EBITDA": ebitda_t,
            "Depreciation": dep_t,
            "EBIT": ebit_t,
            "InterestExpense": interest_t,
            "EBT": ebt_t,
            "Taxes": tax_t,
            "NetIncome": ni_t,
            "Dividends": div_t,
            "AccountsReceivable": ar_end,
            "Inventory": inv_end,
            "AccountsPayable": ap_end,
            "dAR": d_ar,
            "dInventory": d_inv,
            "dAP": d_ap,
            "CFO": cfo_t,
            "CapEx": capex_t,
            "CFI": cfi_t,
            "CFF_BeforeRevolver": cff_before_revolver,
            "RevolverDraw": draw,
            "RevolverRepay": repay,
            "CFF": cff_t,
            "NetChangeCash": net_change_cash_t,
            "Cash_Begin": cash_begin,
            "Cash_End": cash_end,
            "TermDebt_End": term_end,
            "Revolver_End": revl_end,
            "TotalDebt_End": total_debt_end,
            "PPENet_End": ppe_end,
            "RetainedEarnings_End": re_end,
            "CommonCapital_Static": common_capital,
            "OtherAssets_Static": other_assets,
            "OtherLiabilities_Static": other_liab,
            "TotalAssets": total_assets_end,
            "TotalLiabilities": total_liab_end,
            "TotalEquity": total_eq_end,
            "BS_Residual": bs_resid,
            "FCFF": fcff_t,
            "FCFE_Bridge": fcfe_t,
            "NetDebt": net_debt_t,
            "NetDebt_to_EBITDA_Annualized": nde_annual,
        })

        rev_prev = rev_t
        cash_begin = cash_end
        ar_begin = ar_end
        inv_begin = inv_end
        ap_begin = ap_end
        ppe_begin = ppe_end
        term_begin = term_end
        revl_begin = revl_end
        re_begin = re_end

    detail = pd.DataFrame(rows).set_index("Period").sort_index()

    diagnostics = {
        "IndexSortedUnique": bool(detail.index.is_unique and detail.index.is_monotonic_increasing),
        "CashRoll_OK": bool((detail["Cash_End"] - (detail["Cash_Begin"] + detail["CFO"] + detail["CFI"] + detail["CFF"])).abs().max() <= 1e-6),
        "BS_Tie_OK_Within_500": bool(detail["BS_Residual"].abs().max() <= 500.0),
        "Inventory_NonNegative": bool((detail["Inventory"] >= 0.0).all()),
        "Revolver_Within_Limit": bool((detail["Revolver_End"] <= assumptions["revolver_limit"] + 1e-9).all()),
        "CashFloor_Met": bool((detail["Cash_End"] >= assumptions["cash_floor"] - 1e-9).all()),
        "Soft_NearZero_or_Negative_EBITDA": bool((detail["EBITDA"] <= 1.0).any()),
        "Max_BS_Residual": float(detail["BS_Residual"].abs().max()),
        "Max_Cash_Roll_Residual": float((detail["Cash_End"] - (detail["Cash_Begin"] + detail["CFO"] + detail["CFI"] + detail["CFF"])).abs().max()),
    }

    positive_nde = detail["NetDebt_to_EBITDA_Annualized"].dropna()
    summary = pd.Series({
        "EndingCash": float(detail["Cash_End"].iloc[-1]),
        "MinCash": float(detail["Cash_End"].min()),
        "PeakRevolver": float(detail["Revolver_End"].max()),
        "EndingDebt": float(detail["TotalDebt_End"].iloc[-1]),
        "PeakNetDebt_to_EBITDA_Annualized": float(positive_nde.max()) if len(positive_nde) else np.nan,
        "FinalQuarterRevenue": float(detail["Revenue"].iloc[-1]),
        "FinalQuarterEBITDA": float(detail["EBITDA"].iloc[-1]),
    })

    return {
        "IS": detail[["Revenue", "COGS", "GrossProfit", "EBITDA", "Depreciation", "EBIT", "InterestExpense", "EBT", "Taxes", "NetIncome"]],
        "CF": detail[["CFO", "CapEx", "CFI", "Dividends", "CFF_BeforeRevolver", "RevolverDraw", "RevolverRepay", "CFF", "NetChangeCash", "Cash_Begin", "Cash_End", "FCFF", "FCFE_Bridge"]],
        "BS": detail[["Cash_End", "AccountsReceivable", "Inventory", "PPENet_End", "OtherAssets_Static", "TotalAssets", "AccountsPayable", "TermDebt_End", "Revolver_End", "OtherLiabilities_Static", "TotalLiabilities", "RetainedEarnings_End", "CommonCapital_Static", "TotalEquity", "BS_Residual"]],
        "Detail": detail,
        "Summary": summary,
        "Diagnostics": diagnostics,
    }

results = {}
run_manifest_rows = []
for scenario_name in scenario_registry.index:
    assumptions = make_assumptions(base_assumptions, scenario_name, scenario_registry)
    run_id = f"{scenario_name}_{MODEL_VERSION}"
    assumption_set_json = json.dumps({k: float(v) for k, v in assumptions.items()}, sort_keys=True)
    assumption_set_sha256 = hashlib.sha256(assumption_set_json.encode("utf-8")).hexdigest()
    assumption_set_id = f"assumption_set::{scenario_name}::{assumption_set_sha256[:12]}"
    try:
        result = run_three_statement_forecast(hist, assumptions, horizon_q=FORECAST_HORIZON_Q)
        results[scenario_name] = result
        hard_pass = all(result["Diagnostics"][k] for k in ["IndexSortedUnique", "CashRoll_OK", "BS_Tie_OK_Within_500", "Inventory_NonNegative", "Revolver_Within_Limit"])
        soft_warning = bool(result["Diagnostics"]["Soft_NearZero_or_Negative_EBITDA"])
        cash_floor_met = bool(result["Diagnostics"]["CashFloor_Met"])
        diagnostic_status = "pass" if hard_pass and cash_floor_met and not soft_warning else ("warn" if hard_pass else "fail")
        diagnostic_summary = "; ".join([f"{k}={result['Diagnostics'][k]}" for k in sorted(result["Diagnostics"])])
        run_manifest_rows.append({
            "run_id": run_id,
            "company": COMPANY_NAME,
            "model_version": MODEL_VERSION,
            "data_version": DATA_VERSION,
            "data_file": DATA_PATH.name,
            "assumption_set_id": assumption_set_id,
            "assumption_set_json": assumption_set_json,
            "scenario_label": scenario_name,
            "scenario_source": RUN_METADATA["scenario_registry_reference"],
            "diagnostic_status": diagnostic_status,
            "status": "success" if hard_pass else "hard_fail",
            "diagnostic_hard_pass": hard_pass,
            "cash_floor_met": cash_floor_met,
            "soft_warning_near_zero_ebitda": soft_warning,
            "diagnostic_summary": diagnostic_summary,
            "export_bundle_reference": RUN_METADATA["export_bundle_reference"],
            "authoritative_output_reference": RUN_METADATA["authoritative_output_reference"],
            "run_manifest_reference": RUN_METADATA["run_manifest_reference"],
            "run_timestamp": RUN_METADATA["run_timestamp"],
        })
    except Exception as exc:
        results[scenario_name] = {"ERROR": str(exc)}
        run_manifest_rows.append({
            "run_id": run_id,
            "company": COMPANY_NAME,
            "model_version": MODEL_VERSION,
            "data_version": DATA_VERSION,
            "data_file": DATA_PATH.name,
            "assumption_set_id": assumption_set_id,
            "assumption_set_json": assumption_set_json,
            "scenario_label": scenario_name,
            "scenario_source": RUN_METADATA["scenario_registry_reference"],
            "diagnostic_status": "fail",
            "status": "execution_fail",
            "diagnostic_hard_pass": False,
            "cash_floor_met": False,
            "soft_warning_near_zero_ebitda": True,
            "diagnostic_summary": f"execution_error={str(exc)}",
            "export_bundle_reference": RUN_METADATA["export_bundle_reference"],
            "authoritative_output_reference": RUN_METADATA["authoritative_output_reference"],
            "run_manifest_reference": RUN_METADATA["run_manifest_reference"],
            "run_timestamp": RUN_METADATA["run_timestamp"],
        })

run_manifest = pd.DataFrame(run_manifest_rows)
run_manifest = run_manifest[[
    "run_id", "company", "model_version", "data_version", "data_file",
    "assumption_set_id", "assumption_set_json", "scenario_label", "scenario_source",
    "diagnostic_status", "status", "diagnostic_hard_pass", "cash_floor_met",
    "soft_warning_near_zero_ebitda", "diagnostic_summary", "export_bundle_reference",
    "authoritative_output_reference", "run_manifest_reference", "run_timestamp"
]].sort_values(["scenario_label"]).reset_index(drop=True)

comparison_rows = []
diagnostic_rows = []
authoritative_rows = []

for scenario_name, result in results.items():
    if "ERROR" in result:
        diagnostic_rows.append({"scenario": scenario_name, "check": "execution", "value": False, "detail": result["ERROR"]})
        continue

    for metric, value in result["Summary"].items():
        comparison_rows.append({"scenario": scenario_name, "metric": metric, "value": float(value), "model_version": MODEL_VERSION})

    for check, value in result["Diagnostics"].items():
        diagnostic_rows.append({"scenario": scenario_name, "check": check, "value": value, "detail": ""})

    for statement_name, df_stmt in [("IS", result["IS"]), ("CF", result["CF"]), ("BS", result["BS"])]:
        long_df = df_stmt.reset_index().melt(id_vars="Period", var_name="line_item", value_name="value")
        long_df["scenario"] = scenario_name
        long_df["statement"] = statement_name
        long_df["units"] = "USD mm"
        long_df["model_version"] = MODEL_VERSION
        authoritative_rows.append(long_df)

comparison_long = pd.DataFrame(comparison_rows)
comparison_pivot = comparison_long.pivot(index="metric", columns="scenario", values="value").sort_index() if len(comparison_long) else pd.DataFrame()
diagnostic_report = pd.DataFrame(diagnostic_rows)
authoritative_outputs_long = pd.concat(authoritative_rows, ignore_index=True) if authoritative_rows else pd.DataFrame()

display(run_manifest)
print("\nScenario summary (selected metrics)")
display(comparison_pivot.round(2))

print("\nBase scenario — last 6 forecast quarters (detail)")
display(show_recent(results["base"]["Detail"][["Revenue","EBITDA","CFO","Cash_End","Revolver_End","BS_Residual"]].round(2)))


,run_id,company,model_version,data_version,data_file,assumption_set_id,assumption_set_json,scenario_label,scenario_source,diagnostic_status,status,diagnostic_hard_pass,cash_floor_met,soft_warning_near_zero_ebitda,diagnostic_summary,export_bundle_reference,authoritative_output_reference,run_manifest_reference,run_timestamp
0,base_boeing_capstone_combined_v1,The Boeing Company,boeing_capstone_combined_v1,BA_quarterly_statements.xlsx|sha256=469a396bc6...,BA_quarterly_statements.xlsx,assumption_set::base::cffaed30b7e0,"{""borrowing_spread_annual"": 0.0122921328776066...",base,outputs/scenario_registry.csv,warn,success,True,False,False,BS_Tie_OK_Within_500=True; CashFloor_Met=False...,outputs/boeing_capstone_review_pack.xlsx,outputs/authoritative_outputs_long.csv,outputs/run_manifest.csv,2026-04-19T11:47:54
1,downside_boeing_capstone_combined_v1,The Boeing Company,boeing_capstone_combined_v1,BA_quarterly_statements.xlsx|sha256=469a396bc6...,BA_quarterly_statements.xlsx,assumption_set::downside::12eec78181ab,"{""borrowing_spread_annual"": 0.0222921328776066...",downside,outputs/scenario_registry.csv,warn,success,True,False,True,BS_Tie_OK_Within_500=True; CashFloor_Met=False...,outputs/boeing_capstone_review_pack.xlsx,outputs/authoritative_outputs_long.csv,outputs/run_manifest.csv,2026-04-19T11:47:54
2,liquidity_stress_boeing_capstone_combined_v1,The Boeing Company,boeing_capstone_combined_v1,BA_quarterly_statements.xlsx|sha256=469a396bc6...,BA_quarterly_statements.xlsx,assumption_set::liquidity_stress::ed803a1614af,"{""borrowing_spread_annual"": 0.0322921328776066...",liquidity_stress,outputs/scenario_registry.csv,warn,success,True,False,True,BS_Tie_OK_Within_500=True; CashFloor_Met=False...,outputs/boeing_capstone_review_pack.xlsx,outputs/authoritative_outputs_long.csv,outputs/run_manifest.csv,2026-04-19T11:47:54
3,upside_boeing_capstone_combined_v1,The Boeing Company,boeing_capstone_combined_v1,BA_quarterly_statements.xlsx|sha256=469a396bc6...,BA_quarterly_statements.xlsx,assumption_set::upside::b10c8549170d,"{""borrowing_spread_annual"": 0.0097921328776066...",upside,outputs/scenario_registry.csv,warn,success,True,False,False,BS_Tie_OK_Within_500=True; CashFloor_Met=False...,outputs/boeing_capstone_review_pack.xlsx,outputs/authoritative_outputs_long.csv,outputs/run_manifest.csv,2026-04-19T11:47:54



Scenario summary (selected metrics)


scenario,base,downside,liquidity_stress,upside
metric,,,,
EndingCash,"-9,635.38","-9,697.79","-29,715.20","-3,959.39"
EndingDebt,"60,637.00","60,637.00","60,637.00","60,637.00"
FinalQuarterEBITDA,144.92,"-1,035.06","-1,919.43","1,988.82"
FinalQuarterRevenue,"28,983.94","18,819.22","14,764.84","39,776.41"
MinCash,"-9,635.38","-9,697.79","-29,715.20","-3,959.39"
PeakNetDebt_to_EBITDA_Annualized,121.23,NaN,NaN,8.12
PeakRevolver,"15,000.00","15,000.00","15,000.00","15,000.00"



Base scenario — last 6 forecast quarters (detail)


,Revenue,EBITDA,CFO,Cash_End,Revolver_End,BS_Residual
Period,,,,,,
2028-09-30,"27,305.80",136.53,-677.77,0.00,"11,876.37",-0.00
2028-12-31,"27,633.47",138.17,"-1,605.47",0.00,"14,355.24",-0.00
2029-03-31,"27,965.07",139.83,"-3,529.07","-3,768.18","15,000.00",-0.00
2029-06-30,"28,300.65",141.50,-727.27,"-5,389.94","15,000.00",-0.00
2029-09-30,"28,640.26",143.20,-736.11,"-7,031.27","15,000.00",-0.00
2029-12-31,"28,983.94",144.92,"-1,688.02","-9,635.38","15,000.00",0.00


## 6) Two-way sensitivity and DSO break-even


In [7]:

price_shocks = [-0.02, 0.00, 0.02]
volume_shocks = [-0.04, 0.00, 0.04]
grid_records = []

for ps in price_shocks:
    for vs in volume_shocks:
        a = deepcopy(base_assumptions)
        a["price_growth"] = a["price_growth"] + ps
        a["volume_growth"] = a["volume_growth"] + vs
        out = run_three_statement_forecast(hist, a, horizon_q=FORECAST_HORIZON_Q)
        grid_records.append({
            "price_shock": ps,
            "volume_shock": vs,
            "EndingCash": out["Summary"]["EndingCash"],
            "PeakRevolver": out["Summary"]["PeakRevolver"],
            "PeakNetDebt_to_EBITDA_Annualized": out["Summary"]["PeakNetDebt_to_EBITDA_Annualized"],
            "CashFloor_Met": out["Diagnostics"]["CashFloor_Met"],
        })

sensitivity_long = pd.DataFrame(grid_records)
ending_cash_grid = sensitivity_long.pivot(index="volume_shock", columns="price_shock", values="EndingCash")
peak_revolver_grid = sensitivity_long.pivot(index="volume_shock", columns="price_shock", values="PeakRevolver")

def max_dso_without_revolver(hist: pd.DataFrame, assumptions: dict, *, horizon_q: int, dso_low: float, dso_high: float, tol_days: float = 0.25):
    def ok(test_dso):
        a = deepcopy(assumptions)
        a["dso"] = test_dso
        out = run_three_statement_forecast(hist, a, horizon_q=horizon_q)
        return float(out["Summary"]["PeakRevolver"]) <= 1e-6
    lo, hi = dso_low, dso_high
    if not ok(lo):
        return np.nan
    if ok(hi):
        return hi
    while (hi - lo) > tol_days:
        mid = (lo + hi) / 2.0
        if ok(mid):
            lo = mid
        else:
            hi = mid
    return lo

stress_assumptions = make_assumptions(base_assumptions, "liquidity_stress", scenario_registry)
dso_break_even = max_dso_without_revolver(
    hist,
    stress_assumptions,
    horizon_q=FORECAST_HORIZON_Q,
    dso_low=max(0.0, stress_assumptions["dso"] - 40.0),
    dso_high=stress_assumptions["dso"] + 40.0,
)

print("Ending cash sensitivity grid ($mm)")
display(ending_cash_grid.round(2))

print("\nPeak revolver sensitivity grid ($mm)")
display(peak_revolver_grid.round(2))

print("\nLiquidity-stress DSO break-even (max DSO with no revolver draw)")
display(pd.DataFrame({"LiquidityStress_DSO_BreakEven_MaxDays": [dso_break_even]}))


Ending cash sensitivity grid ($mm)


price_shock,-0.02,0.00,0.02
volume_shock,,,
-0.04,"-2,006.11","-29,723.23","-65,636.79"
0.00,"-64,881.97","-113,901.02","-178,808.14"
0.04,"-171,798.21","-259,009.35","-375,022.32"



Peak revolver sensitivity grid ($mm)


price_shock,-0.02,0.00,0.02
volume_shock,,,
-0.04,"15,000.00","15,000.00","15,000.00"
0.00,"15,000.00","15,000.00","15,000.00"
0.04,"15,000.00","15,000.00","15,000.00"



Liquidity-stress DSO break-even (max DSO with no revolver draw)


,LiquidityStress_DSO_BreakEven_MaxDays
0,NaN


## 7) Final Reviewer Summary

Quick pass/fail check + failure detail report


In [8]:
def _status_label(ok: bool) -> str:
    return "PASS" if bool(ok) else "FAIL"

def _fmt_num(x):
    try:
        return f"{float(x):,.2f}"
    except Exception:
        return str(x)

def _get_series(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(index=df.index, dtype=float)

def _show_detail(title, df):
    print(f"\n{title}")
    if df is None or df.empty:
        print("No rows to display.")
    else:
        display(df.round(2) if hasattr(df, "round") else df)

def _cols_that_exist(df, candidates):
    return [c for c in candidates if c in df.columns]

# Combine for easier diagnostics
diag_all = pd.concat([hist.copy(), statement_checks.copy()], axis=1)

# ----------------------------
# Core historical integrity checks
# ----------------------------
bs_tie_max = float(_get_series(statement_checks, "BS_Tie_Residual").abs().max())
cf_roll_max = float(_get_series(statement_checks, "CF_Roll_Residual_Detailed").abs().max())
gp_resid_max = float(_get_series(statement_checks, "GrossProfit_Residual").abs().max())
ebitda_resid_max = float(_get_series(statement_checks, "EBITDA_Residual").abs().max())
re_roll_max = float(_get_series(statement_checks, "RE_Roll_Residual").abs().max())

bs_tie_ok = bs_tie_max <= 1e-6
cf_roll_ok = cf_roll_max <= 50.0
gp_ok = gp_resid_max <= 1e-6
ebitda_ok = ebitda_resid_max <= 100.0
re_ok = re_roll_max <= 100.0

# ----------------------------
# Scenario / diagnostics summary
# ----------------------------
scenario_fail_count = int((run_manifest["status"] == "hard_fail").sum()) if "status" in run_manifest.columns else 0
scenario_ok = scenario_fail_count == 0

# ----------------------------
# Valuation / market data presence
# ----------------------------
valuation_ok = ("PricePerShare" in hist.columns) and hist["PricePerShare"].notna().all()

# ----------------------------
# Simplified roll-forward residuals (informational only)
# ----------------------------
ppe_roll_max = float(_get_series(statement_checks, "PP&E_Roll_Residual").abs().max())
debt_roll_max = float(_get_series(statement_checks, "Debt_Roll_Residual").abs().max())
rollforward_disclosure_ok = True

# ----------------------------
# Export bundle readiness
# ----------------------------
expected_export_files = [
    "historical_driver_pack.csv",
    "common_size_income_statement.csv",
    "common_size_balance_sheet.csv",
    "historical_ratios.csv",
    "historical_trends.csv",
    "historical_exception_report.csv",
    "statement_integrity_checks.csv",
    "cfo_bridge.csv",
    "free_cash_flow_view.csv",
    "scenario_registry.csv",
    "run_manifest.csv",
    "scenario_summary_long.csv",
    "scenario_summary_pivot.csv",
    "scenario_diagnostic_report.csv",
    "authoritative_outputs_long.csv",
    "two_way_sensitivity_long.csv",
    "system_checks.csv",
    "run_metadata.json",
    "boeing_capstone_review_pack.xlsx",
]
export_ready_ok = OUTPUT_DIR.exists()

reviewer_summary = pd.DataFrame([
    {
        "Check": "Source file located",
        "Status": _status_label(DATA_PATH.exists()),
        "Detail": str(DATA_PATH)
    },
    {
        "Check": "Historical balance-sheet tie-out",
        "Status": _status_label(bs_tie_ok),
        "Detail": f"Max abs residual = {_fmt_num(bs_tie_max)}"
    },
    {
        "Check": "Historical cash roll-forward",
        "Status": _status_label(cf_roll_ok),
        "Detail": f"Max abs residual = {_fmt_num(cf_roll_max)}"
    },
    {
        "Check": "Historical gross profit bridge",
        "Status": _status_label(gp_ok),
        "Detail": f"Max abs residual = {_fmt_num(gp_resid_max)}"
    },
    {
        "Check": "Historical EBITDA bridge",
        "Status": _status_label(ebitda_ok),
        "Detail": f"Max abs residual = {_fmt_num(ebitda_resid_max)}"
    },
    {
        "Check": "Retained earnings roll-forward",
        "Status": _status_label(re_ok),
        "Detail": f"Max abs residual = {_fmt_num(re_roll_max)}"
    },
    {
        "Check": "Named scenarios executed",
        "Status": _status_label(scenario_ok),
        "Detail": f"Hard-fail scenarios = {scenario_fail_count}"
    },
    {
        "Check": "Valuation / market data populated",
        "Status": _status_label(valuation_ok),
        "Detail": f"Non-null quarter prices = {int(hist['PricePerShare'].notna().sum()) if 'PricePerShare' in hist.columns else 0} of {len(hist)}"
    },
    {
        "Check": "PP&E and debt roll-forward residuals disclosed",
        "Status": _status_label(rollforward_disclosure_ok),
        "Detail": f"Analytical residuals retained for transparency | PP&E max abs = {_fmt_num(ppe_roll_max)} | Debt max abs = {_fmt_num(debt_roll_max)}"
    },
    {
        "Check": "Export bundle ready",
        "Status": _status_label(export_ready_ok),
        "Detail": f"Next cell writes {len(expected_export_files)} files to {OUTPUT_DIR}"
    },
], columns=["Check", "Status", "Detail"])

all_core_pass = all([
    DATA_PATH.exists(),
    bs_tie_ok,
    cf_roll_ok,
    gp_ok,
    ebitda_ok,
    re_ok,
    scenario_ok,
    valuation_ok,
    export_ready_ok
])

print("FINAL REVIEWER SUMMARY — QUICK CHECK")
print(f"Company: {COMPANY_NAME}")
print(f"Model version: {MODEL_VERSION}")
print(f"Data version: {DATA_VERSION}")
print(f"Run timestamp: {RUN_METADATA['run_timestamp']}")
print(f"Overall core status: {'PASS' if all_core_pass else 'CHECK ITEMS BELOW'}")

display(reviewer_summary)

# ============================================================
# IF anything fails, show the exact quarters / rows involved
# ============================================================
if not all_core_pass:
    print("\nDETAILED FAILURE OUTPUT")

    # 1) Source file
    if not DATA_PATH.exists():
        print("\n[Source file located] FAIL")
        print(f"Could not find source file at: {DATA_PATH}")

    # 2) Balance-sheet tie-out
    if not bs_tie_ok and "BS_Tie_Residual" in statement_checks.columns:
        cols = _cols_that_exist(diag_all, [
            "BS_Tie_Residual",
            "TotalAssets", "TotalLiabilities", "TotalEquity",
            "TotalDebt", "Cash", "RetainedEarnings"
        ])
        fail_df = diag_all.loc[diag_all["BS_Tie_Residual"].abs() > 1e-6, cols].copy()
        fail_df["Abs_BS_Tie_Residual"] = fail_df["BS_Tie_Residual"].abs()
        fail_df = fail_df.sort_values("Abs_BS_Tie_Residual", ascending=False)
        _show_detail("[Historical balance-sheet tie-out] FAIL — offending quarters", fail_df)

    # 3) Cash roll-forward
    if not cf_roll_ok and "CF_Roll_Residual_Detailed" in statement_checks.columns:
        cols = _cols_that_exist(diag_all, [
            "CF_Roll_Residual_Detailed", "CF_Roll_Residual_vs_NetChange",
            "Cash", "CashAndEquivalents",
            "CFO", "CFI", "CFF", "OtherCash",
            "NetChangeInCash", "DeltaCash_Reported"
        ])
        fail_df = diag_all.loc[diag_all["CF_Roll_Residual_Detailed"].abs() > 50.0, cols].copy()
        fail_df["Abs_CF_Roll_Residual"] = fail_df["CF_Roll_Residual_Detailed"].abs()
        fail_df = fail_df.sort_values("Abs_CF_Roll_Residual", ascending=False)
        _show_detail("[Historical cash roll-forward] FAIL — offending quarters", fail_df)

    # 4) Gross profit bridge
    if not gp_ok and "GrossProfit_Residual" in statement_checks.columns:
        cols = _cols_that_exist(diag_all, [
            "GrossProfit_Residual",
            "Revenue", "COGS", "GrossProfit"
        ])
        fail_df = diag_all.loc[diag_all["GrossProfit_Residual"].abs() > 1e-6, cols].copy()
        fail_df["Abs_GrossProfit_Residual"] = fail_df["GrossProfit_Residual"].abs()
        fail_df = fail_df.sort_values("Abs_GrossProfit_Residual", ascending=False)
        _show_detail("[Historical gross profit bridge] FAIL — offending quarters", fail_df)

    # 5) EBITDA bridge
    if not ebitda_ok and "EBITDA_Residual" in statement_checks.columns:
        cols = _cols_that_exist(diag_all, [
            "EBITDA_Residual",
            "Revenue", "GrossProfit", "EBIT", "OperatingIncome",
            "D&A", "Depreciation", "Amortization", "EBITDA_Reported"
        ])
        fail_df = diag_all.loc[diag_all["EBITDA_Residual"].abs() > 100.0, cols].copy()
        fail_df["Abs_EBITDA_Residual"] = fail_df["EBITDA_Residual"].abs()
        fail_df = fail_df.sort_values("Abs_EBITDA_Residual", ascending=False)
        _show_detail("[Historical EBITDA bridge] FAIL — offending quarters", fail_df)

    # 6) Retained earnings roll-forward
    if not re_ok and "RE_Roll_Residual" in statement_checks.columns:
        re_cols = _cols_that_exist(diag_all, ["RetainedEarnings", "NetIncome", "DividendsPaid_Reported"])
        fail_df = diag_all.loc[diag_all["RE_Roll_Residual"].abs() > 100.0, re_cols + ["RE_Roll_Residual"]].copy()

        if "RetainedEarnings" in diag_all.columns:
            fail_df["RE_Begin"] = diag_all["RetainedEarnings"].shift(1).reindex(fail_df.index)
        else:
            fail_df["RE_Begin"] = pd.NA

        if set(["RetainedEarnings", "NetIncome", "DividendsPaid_Reported"]).issubset(diag_all.columns):
            fail_df["RE_Rebuilt"] = (
                diag_all["RetainedEarnings"].shift(1)
                + diag_all["NetIncome"]
                - diag_all["DividendsPaid_Reported"]
            ).reindex(fail_df.index)
        else:
            fail_df["RE_Rebuilt"] = pd.NA

        fail_df["Abs_RE_Roll_Residual"] = fail_df["RE_Roll_Residual"].abs()
        ordered = ["RE_Begin", "NetIncome", "DividendsPaid_Reported", "RE_Rebuilt", "RetainedEarnings", "RE_Roll_Residual", "Abs_RE_Roll_Residual"]
        ordered = [c for c in ordered if c in fail_df.columns]
        fail_df = fail_df[ordered].sort_values("Abs_RE_Roll_Residual", ascending=False)
        _show_detail("[Retained earnings roll-forward] FAIL — offending quarters", fail_df)

    # 7) Scenario failures
    if not scenario_ok:
        if "status" in run_manifest.columns:
            fail_df = run_manifest.loc[run_manifest["status"] == "hard_fail"].copy()
            _show_detail("[Named scenarios executed] FAIL — failed scenario rows", fail_df)
        else:
            print("\n[Named scenarios executed] FAIL")
            print("run_manifest does not contain a 'status' column.")

    # 8) Valuation / market data failures
    if not valuation_ok:
        val_cols = _cols_that_exist(diag_all, [
            "PricePerShare", "SharesUsedForMarketCap", "MarketCap",
            "TotalDebt", "Cash", "EnterpriseValue"
        ])
        if "PricePerShare" in diag_all.columns:
            fail_df = diag_all.loc[diag_all["PricePerShare"].isna(), val_cols].copy()
            _show_detail("[Valuation / market data populated] FAIL — quarters with missing price data", fail_df)
        else:
            print("\n[Valuation / market data populated] FAIL")
            print("Column 'PricePerShare' not found in hist.")

    # 9) Export readiness
    if not export_ready_ok:
        print("\n[Export bundle ready] FAIL")
        print(f"Output directory does not exist yet: {OUTPUT_DIR}")

    # 10) Informational note for analytical residuals
    print("\n[Informational] PP&E and debt roll-forward residuals are retained as analytical disclosures, not treated as binding tie-out failures.")
    info_cols = _cols_that_exist(diag_all, ["PP&E_Roll_Residual", "Debt_Roll_Residual", "RE_Roll_Residual"])
    if info_cols:
        info_df = diag_all[info_cols].copy()
        _show_detail("Analytical roll-forward residual view", info_df)

if all_core_pass:
    print("\nReviewer summary result: PASS")

FINAL REVIEWER SUMMARY — QUICK CHECK
Company: The Boeing Company
Model version: boeing_capstone_combined_v1
Data version: BA_quarterly_statements.xlsx|sha256=469a396bc6e3|modified=2026-01-27T15:41:58
Run timestamp: 2026-04-19T11:47:54
Overall core status: CHECK ITEMS BELOW


,Check,Status,Detail
0,Source file located,PASS,C:\Users\pcswe\Fin Modeling\BA_quarterly_state...
1,Historical balance-sheet tie-out,PASS,Max abs residual = 0.00
2,Historical cash roll-forward,PASS,Max abs residual = 41.00
3,Historical gross profit bridge,PASS,Max abs residual = 0.00
4,Historical EBITDA bridge,PASS,Max abs residual = 84.00
5,Retained earnings roll-forward,FAIL,"Max abs residual = 5,307.00"
6,Named scenarios executed,PASS,Hard-fail scenarios = 0
7,Valuation / market data populated,PASS,Non-null quarter prices = 40 of 40
8,PP&E and debt roll-forward residuals disclosed,PASS,Analytical residuals retained for transparency...
9,Export bundle ready,PASS,Next cell writes 19 files to C:\Users\pcswe\Fi...



DETAILED FAILURE OUTPUT

[Retained earnings roll-forward] FAIL — offending quarters


,RE_Begin,NetIncome,DividendsPaid_Reported,RE_Rebuilt,RetainedEarnings,RE_Roll_Residual,Abs_RE_Roll_Residual
Period,,,,,,,
2018-03-31,"45,320.00","2,474.00","1,006.00","46,788.00","52,095.00","-5,307.00","5,307.00"
2018-12-31,"54,666.00","3,422.00",970.00,"57,118.00","55,941.00","1,177.00","1,177.00"
2019-12-31,"53,986.00","-1,010.00","1,157.00","51,819.00","50,644.00","1,175.00","1,175.00"
2019-06-30,"58,090.00","-2,942.00","1,156.00","53,992.00","52,819.00","1,173.00","1,173.00"
2019-03-31,"55,941.00","2,147.00","1,161.00","56,927.00","58,090.00","-1,163.00","1,163.00"
2019-09-30,"52,819.00","1,166.00","1,156.00","52,829.00","53,986.00","-1,157.00","1,157.00"
2017-12-31,"44,052.00","3,130.00",842.00,"46,340.00","45,320.00","1,020.00","1,020.00"
2018-06-30,"52,095.00","2,196.00",991.00,"53,300.00","52,303.00",997.00,997.00
2020-03-31,"50,644.00",-628.00,"1,158.00","48,858.00","49,854.00",-996.00,996.00



[Informational] PP&E and debt roll-forward residuals are retained as analytical disclosures, not treated as binding tie-out failures.

Analytical roll-forward residual view


,PP&E_Roll_Residual,Debt_Roll_Residual,RE_Roll_Residual
Period,,,
2016-03-31,NaN,NaN,NaN
2016-06-30,-40.00,18.00,688.00
2016-09-30,-59.00,-10.00,-679.00
2016-12-31,-41.00,-74.00,886.00
2017-03-31,-40.00,48.00,-870.00
2017-06-30,98.00,4.00,840.00
2017-09-30,-15.00,-18.00,-877.00
2017-12-31,-107.00,-18.00,"1,020.00"
2018-03-31,-43.00,-19.00,"-5,307.00"


## 8) Export pack and final diagnostics / system checks

This final section writes the key outputs to the `outputs` folder and then reports the final system checks. The point is not to force green lights when the source data themselves contain definition noise; the point is to preserve the evidence.


In [9]:

system_checks = pd.DataFrame([
    {"check": "source_file_found", "status": DATA_PATH.exists(), "detail": str(DATA_PATH)},
    {"check": "three_statement_rows_match", "status": len(income_q) == len(balance_q) == len(cashflow_q), "detail": f"IS={len(income_q)}, BS={len(balance_q)}, CF={len(cashflow_q)}"},
    {"check": "historical_bs_tie_max_abs_zero", "status": statement_checks["BS_Tie_Residual"].abs().max() <= 1e-6, "detail": float(statement_checks["BS_Tie_Residual"].abs().max())},
    {"check": "historical_cf_roll_max_abs_le_50", "status": statement_checks["CF_Roll_Residual_Detailed"].abs().max() <= 50.0, "detail": float(statement_checks["CF_Roll_Residual_Detailed"].abs().max())},
    {"check": "historical_gross_profit_residual_zero", "status": statement_checks["GrossProfit_Residual"].abs().max() <= 1e-6, "detail": float(statement_checks["GrossProfit_Residual"].abs().max())},
    {"check": "historical_ebitda_residual_le_100", "status": statement_checks["EBITDA_Residual"].abs().max() <= 100.0, "detail": float(statement_checks["EBITDA_Residual"].abs().max())},
    {"check": "all_named_scenarios_executed", "status": (run_manifest["status"] != "execution_fail").all(), "detail": run_manifest["status"].to_dict()},
    {"check": "authoritative_outputs_created", "status": len(authoritative_outputs_long) > 0, "detail": int(len(authoritative_outputs_long))},
    {"check": "valuation_data_available", "status": hist["PricePerShare"].notna().any(), "detail": "offline-safe fallback may leave valuation multiples blank"},
])

historical_driver_pack_path = OUTPUT_DIR / "historical_driver_pack.csv"
common_size_is_path = OUTPUT_DIR / "common_size_income_statement.csv"
common_size_bs_path = OUTPUT_DIR / "common_size_balance_sheet.csv"
ratio_path = OUTPUT_DIR / "historical_ratios.csv"
trend_path = OUTPUT_DIR / "historical_trends.csv"
exception_path = OUTPUT_DIR / "historical_exception_report.csv"
statement_check_path = OUTPUT_DIR / "statement_integrity_checks.csv"
cfo_bridge_path = OUTPUT_DIR / "cfo_bridge.csv"
fcf_path = OUTPUT_DIR / "free_cash_flow_view.csv"
scenario_registry_path = OUTPUT_DIR / "scenario_registry.csv"
run_manifest_path = OUTPUT_DIR / "run_manifest.csv"
comparison_long_path = OUTPUT_DIR / "scenario_summary_long.csv"
comparison_pivot_path = OUTPUT_DIR / "scenario_summary_pivot.csv"
diagnostic_report_path = OUTPUT_DIR / "scenario_diagnostic_report.csv"
authoritative_path = OUTPUT_DIR / "authoritative_outputs_long.csv"
sensitivity_path = OUTPUT_DIR / "two_way_sensitivity_long.csv"
system_check_path = OUTPUT_DIR / "system_checks.csv"
metadata_path = OUTPUT_DIR / "run_metadata.json"
review_pack_path = OUTPUT_DIR / "boeing_capstone_review_pack.xlsx"
change_log_path = OUTPUT_DIR / CHANGE_LOG_NAME

hist.to_csv(historical_driver_pack_path)
common_size_is.to_csv(common_size_is_path)
common_size_bs.to_csv(common_size_bs_path)
ratios.to_csv(ratio_path)
trend_df.to_csv(trend_path)
exception_report.to_csv(exception_path)
statement_checks.to_csv(statement_check_path)
cfo_bridge.to_csv(cfo_bridge_path)
fcf_view.to_csv(fcf_path)
scenario_registry.to_csv(scenario_registry_path)
run_manifest.to_csv(run_manifest_path, index=False)
comparison_long.to_csv(comparison_long_path, index=False)
comparison_pivot.to_csv(comparison_pivot_path)
diagnostic_report.to_csv(diagnostic_report_path, index=False)
authoritative_outputs_long.to_csv(authoritative_path, index=False)
sensitivity_long.to_csv(sensitivity_path, index=False)
system_checks.to_csv(system_check_path, index=False)
CHANGE_LOG.to_csv(change_log_path, index=False)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(RUN_METADATA, f, indent=2)

with pd.ExcelWriter(review_pack_path, engine="openpyxl") as writer:
    hist.to_excel(writer, sheet_name="HistoricalDriverPack")
    common_size_is.to_excel(writer, sheet_name="CommonSizeIS")
    common_size_bs.to_excel(writer, sheet_name="CommonSizeBS")
    ratios.to_excel(writer, sheet_name="Ratios")
    trend_df.to_excel(writer, sheet_name="Trends")
    exception_report.to_excel(writer, sheet_name="ExceptionReport")
    statement_checks.to_excel(writer, sheet_name="StatementChecks")
    cfo_bridge.to_excel(writer, sheet_name="CFOBridge")
    fcf_view.to_excel(writer, sheet_name="FCFView")
    scenario_registry.to_excel(writer, sheet_name="ScenarioRegistry")
    run_manifest.to_excel(writer, sheet_name="RunManifest", index=False)
    comparison_pivot.to_excel(writer, sheet_name="ScenarioSummary")
    diagnostic_report.to_excel(writer, sheet_name="ScenarioDiagnostics", index=False)
    authoritative_outputs_long.to_excel(writer, sheet_name="AuthoritativeOutputs", index=False)
    sensitivity_long.to_excel(writer, sheet_name="TwoWaySensitivity", index=False)
    ending_cash_grid.to_excel(writer, sheet_name="TwoWay_EndingCash")
    peak_revolver_grid.to_excel(writer, sheet_name="TwoWay_PeakRevolver")
    pd.DataFrame({"LiquidityStress_DSO_BreakEven_MaxDays": [dso_break_even]}).to_excel(writer, sheet_name="BreakEven", index=False)
    system_checks.to_excel(writer, sheet_name="SystemChecks", index=False)
    CHANGE_LOG.to_excel(writer, sheet_name="ChangeLog", index=False)

display(system_checks)

print("\nRun metadata")
display(pd.Series(RUN_METADATA, name="run_metadata"))

print("\nFormal change log")
display(CHANGE_LOG)

print("\nExported review pack:")
print(review_pack_path)

print("\nOther exported files:")
for p in [
    historical_driver_pack_path, common_size_is_path, common_size_bs_path, ratio_path,
    trend_path, exception_path, statement_check_path, cfo_bridge_path, fcf_path,
    scenario_registry_path, run_manifest_path, comparison_long_path, comparison_pivot_path,
    diagnostic_report_path, authoritative_path, sensitivity_path, system_check_path, metadata_path,
    change_log_path
]:
    print(p)


,check,status,detail
0,source_file_found,True,C:\Users\pcswe\Fin Modeling\BA_quarterly_state...
1,three_statement_rows_match,True,"IS=40, BS=40, CF=40"
2,historical_bs_tie_max_abs_zero,True,0.00
3,historical_cf_roll_max_abs_le_50,True,41.00
4,historical_gross_profit_residual_zero,True,0.00
5,historical_ebitda_residual_le_100,True,84.00
6,all_named_scenarios_executed,True,"{0: 'success', 1: 'success', 2: 'success', 3: ..."
7,authoritative_outputs_created,True,2432
8,valuation_data_available,True,offline-safe fallback may leave valuation mult...



Run metadata


run_timestamp                                                   2026-04-19T11:47:54
company                                                          The Boeing Company
notebook_name                     Boeing_Capstone_Final_Combined_Notebook_Hardco...
model_version                                           boeing_capstone_combined_v1
data_file                                              BA_quarterly_statements.xlsx
data_path                         C:\Users\pcswe\Fin Modeling\BA_quarterly_state...
data_version                      BA_quarterly_statements.xlsx|sha256=469a396bc6...
data_file_sha256                  469a396bc6e3fff3446b0e145f650d5dcfe45cbaa1450a...
data_file_modified_timestamp                                    2026-01-27T15:41:58
scenario_registry_reference                           outputs/scenario_registry.csv
run_manifest_reference                                     outputs/run_manifest.csv
authoritative_output_reference               outputs/authoritative_outputs_l


Formal change log


,change_id,change_timestamp,change_area,change_summary,affected_artifacts,change_type
0,CL-001,2026-04-19T11:47:54,market_data,Added hard-coded quarter-end Boeing market cap...,Setup cell; historical ratios and valuation ou...,logic/input enhancement
1,CL-002,2026-04-19T11:47:54,run_control,Expanded the run manifest to an explicit textb...,Run manifest; exported run manifest CSV; revie...,metadata enhancement
2,CL-003,2026-04-19T11:47:54,documentation,Added a formal change log artifact and export ...,In-notebook change log display; exported chang...,documentation enhancement
3,CL-004,2026-04-19T11:47:54,assumptions,Replaced mechanically-extrapolated scenario re...,scenario_registry; authoritative_outputs_long....,assumption revision
4,CL-005,2026-04-19T11:47:54,model_scope,Extended forecast horizon from 8 quarters (2 y...,"All scenario outputs, authoritative_outputs_lo...",model scope extension



Exported review pack:
C:\Users\pcswe\Fin Modeling\outputs\boeing_capstone_review_pack.xlsx

Other exported files:
C:\Users\pcswe\Fin Modeling\outputs\historical_driver_pack.csv
C:\Users\pcswe\Fin Modeling\outputs\common_size_income_statement.csv
C:\Users\pcswe\Fin Modeling\outputs\common_size_balance_sheet.csv
C:\Users\pcswe\Fin Modeling\outputs\historical_ratios.csv
C:\Users\pcswe\Fin Modeling\outputs\historical_trends.csv
C:\Users\pcswe\Fin Modeling\outputs\historical_exception_report.csv
C:\Users\pcswe\Fin Modeling\outputs\statement_integrity_checks.csv
C:\Users\pcswe\Fin Modeling\outputs\cfo_bridge.csv
C:\Users\pcswe\Fin Modeling\outputs\free_cash_flow_view.csv
C:\Users\pcswe\Fin Modeling\outputs\scenario_registry.csv
C:\Users\pcswe\Fin Modeling\outputs\run_manifest.csv
C:\Users\pcswe\Fin Modeling\outputs\scenario_summary_long.csv
C:\Users\pcswe\Fin Modeling\outputs\scenario_summary_pivot.csv
C:\Users\pcswe\Fin Modeling\outputs\scenario_diagnostic_report.csv
C:\Users\pcswe\Fin Mod